In [0]:
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *


schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("avatar", StringType(), True)
])


base_url = "https://reqres.in/api/users?page={}"
headers = {
    "x-api-key": "reqres_f84e4451a9e645e78c89a38b40af1425"
}

all_data = []

page = 2

while True:
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    json_data = response.json()
    
    data = json_data.get("data", [])
    
    if not data:   # stop condition
        break
    
    all_data.extend(data)
    page += 1
print(all_data)
print("Total records fetched:", len(all_data))
     


df = spark.createDataFrame(all_data, schema=schema)
display(df)
     




In [0]:
df = df.withColumn("site_address", split(col("email"), "@")[1])

display(df)
     

df = df.withColumn("load_date", current_date())
display(df)
     


In [0]:

spark.sql("CREATE DATABASE IF NOT EXISTS site_info")
     

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("api_person_info")
     


In [0]:
%sql
select * from api_person_info